# Validation Test Evaluation (large amplitude study) - Oscillating droplet

This notebook and the corresponding simulation setup notebook (OscillatingDroplet_DACH_Run.ipynb) are part of published results (cf. V.B.) found in *From weakly to strongly nonlinear viscous drop shape oscillations: An analytical and numerical study* (https://doi.org/10.1103/PhysRevFluids.9.063601). 

In [ ]:
#r "BoSSSpad.dll"
//#r "../../../src/L4-application/BoSSSpad/bin/Release/net8.0/BoSSSpad.dll"
using System;
using System.Collections.Generic;
using System.Linq;
using ilPSP;
using ilPSP.Utils;
using BoSSS.Platform;
using BoSSS.Foundation;
using BoSSS.Foundation.XDG;
using BoSSS.Foundation.Grid;
using BoSSS.Foundation.Grid.Classic;
using BoSSS.Foundation.IO;
using BoSSS.Solution;
using BoSSS.Solution.Control;
using BoSSS.Solution.GridImport;
using BoSSS.Solution.Statistic;
using BoSSS.Solution.Utils;
using BoSSS.Solution.AdvancedSolvers;
using BoSSS.Solution.Gnuplot;
using BoSSS.Solution.LevelSetTools;
using BoSSS.Solution.LevelSetTools.SolverWithLevelSetUpdater;
using BoSSS.Solution.NSECommon;
using BoSSS.Solution.XNSECommon;
using BoSSS.Solution.XdgTimestepping;
using BoSSS.Application.BoSSSpad;
using BoSSS.Application.XNSE_Solver;
using static BoSSS.Application.BoSSSpad.BoSSSshell;
Init();

In [ ]:
BoSSSshell.WorkflowMgm.Init("OscillatingDropletPaper");

Opening existing database 'C:\BoSSS-localJobs\OscillatingDropletPaper_OnJenkins'.


In [3]:
using System.IO;
using BoSSS.Foundation.Quadrature;
using BoSSS.Solution.XNSECommon;
using BoSSS.Application.XNSE_Solver.Logging;
using BoSSS.Application.XNSE_Solver.PhysicalBasedTestcases;

## Sessions

In [ ]:
var sessions = BoSSSshell.WorkflowMgm.Sessions.ToList();
//var sessions = databases.Pick(0).Sessions;
sessions

#0: OscillatingDropletPaper	OD3D_m2_Oh01_eta06_stagnantInit	12/6/2025 10:30:45 PM	10651a2d...
#1: OscillatingDropletPaper	OD3D_m4_Oh01_eta02_stagnantInit	12/6/2025 10:31:35 PM	b2ad17f9...
#2: OscillatingDropletPaper	OD3D_m4_Oh01_eta03_stagnantInit	12/6/2025 10:31:48 PM	1a466c4b...
#3: OscillatingDropletPaper	OD3D_m2_Oh01_eta07_stagnantInit	12/6/2025 10:30:57 PM	9b83588b...
#4: OscillatingDropletPaper	OD3D_m2_Oh01_eta03_stagnantInit	12/6/2025 10:30:22 PM	42c06632...
#5: OscillatingDropletPaper	OD3D_m4_Oh01_eta06_stagnantInit	12/6/2025 10:32:15 PM	f20038ae...
#6: OscillatingDropletPaper	OD3D_m3_Oh01_eta05_stagnantInit*	12/6/2025 10:31:22 PM	5bd0a0e3...
#7: OscillatingDropletPaper	OD3D_m2_Oh01_eta05_stagnantInit	12/6/2025 10:30:33 PM	abb9c251...
#8: OscillatingDropletPaper	OD3D_m4_Oh01_eta07_stagnantInit*	12/6/2025 10:32:31 PM	e1f7fb46...
#9: OscillatingDropletPaper	OD3D_m3_Oh01_eta03_stagnantInit	12/6/2025 10:31:10 PM	ca424e65...
#10: OscillatingDropletPaper	OD3D_m4_Oh01_eta05_stagnantIn

In [22]:
bool runShortTests = true; // set to 'false' to run all test cases for the whole period of time (up tp 7.0), otherwise they will run only up to 0.5

## Large amplitude study

In [23]:
string[] mOhS = { "m2", "m3", "m4" };
List<string>[] studyCases = new List<string>[mOhS.Length];
studyCases[0] = new List<string>() { "01", "02", "03", "04", "05", "06", "07" };
studyCases[1] = new List<string>() { "015", "03", "04", "05" };
studyCases[2] = new List<string>() { "01", "02", "03", "04", "05", "06", "07" };

In [ ]:
// origin database including all sessions, in case of missing sessions in wmg, due to rerunning only a subset of all needed runs (reduced computational cost)
var originDB = OpenOrCreateDatabase(@"\\fdygitrunner\ValidationTests\databases\bkup-2025Dez12_000000.OscillatingDropletPaper");

Opening existing database '\\dc3\userspace\smuda\Databases\OscillatingDropletPaper'.


In [25]:
List<ISessionInfo> evalSessOrigin = new List<ISessionInfo>();

In [26]:
for (int i = 0; i < mOhS.Length; i++) {
foreach (string etaCase in studyCases[i]) {

    string studyName = $"OD3D_{mOhS[i]}_Oh01_eta{etaCase}_stagnantInit";
    Console.WriteLine("Searching for: " + studyName);
    if (mOhS[i] == "m2" && etaCase == "05") {
        studyName = $"OD3D_{mOhS[i]}_Oh01_eta{etaCase}_stagnantInit_Rerun";
    }
    var studySess = originDB.Sessions.Where(sess => sess.Name.Contains(studyName)); // && !sess.Name.Contains("Rerun"));

    int studyCount = studySess.Count();
    if (studyCount == 0) {
        Console.WriteLine("Not found!");
    } else if (studyCount > 1) {
        Console.WriteLine("Restart session found! Take last run");       
        evalSessOrigin.Add(studySess.Where(sess => sess.Name.EndsWith($"_restart{studyCount-1}")).Single());
    } else {
        evalSessOrigin.Add(studySess.Single());
        Console.WriteLine("found");
    }

}
}
evalSessOrigin

Searching for: OD3D_m2_Oh01_eta01_stagnantInit
Restart session found! Take last run
Searching for: OD3D_m2_Oh01_eta02_stagnantInit
Restart session found! Take last run
Searching for: OD3D_m2_Oh01_eta03_stagnantInit
Restart session found! Take last run
Searching for: OD3D_m2_Oh01_eta04_stagnantInit
Restart session found! Take last run
Searching for: OD3D_m2_Oh01_eta05_stagnantInit
Restart session found! Take last run
Searching for: OD3D_m2_Oh01_eta06_stagnantInit
Restart session found! Take last run
Searching for: OD3D_m2_Oh01_eta07_stagnantInit
Restart session found! Take last run
Searching for: OD3D_m3_Oh01_eta015_stagnantInit
Restart session found! Take last run
Searching for: OD3D_m3_Oh01_eta03_stagnantInit
Restart session found! Take last run
Searching for: OD3D_m3_Oh01_eta04_stagnantInit
Restart session found! Take last run
Searching for: OD3D_m3_Oh01_eta05_stagnantInit
Restart session found! Take last run
Searching for: OD3D_m4_Oh01_eta01_stagnantInit
Restart session found! Take 

#0: OscillatingDropletPaper	OD3D_m2_Oh01_eta01_stagnantInit_restart3	6/15/2022 8:37:41 AM	dd97e2f1...
#1: OscillatingDropletPaper	OD3D_m2_Oh01_eta02_stagnantInit_restart3	6/15/2022 8:37:47 AM	95fc59fd...
#2: OscillatingDropletPaper	OD3D_m2_Oh01_eta03_stagnantInit_restart1*	5/20/2022 4:43:34 PM	72f1baaa...
#3: OscillatingDropletPaper	OD3D_m2_Oh01_eta04_stagnantInit_restart3	6/13/2022 3:31:38 PM	ceb7f024...
#4: OscillatingDropletPaper	OD3D_m2_Oh01_eta05_stagnantInit_Rerun_restart1	4/7/2023 6:47:46 PM	4006953c...
#5: OscillatingDropletPaper	OD3D_m2_Oh01_eta06_stagnantInit_restart2	6/15/2022 9:32:49 AM	43a86b55...
#6: OscillatingDropletPaper	OD3D_m2_Oh01_eta07_stagnantInit_restart2	6/13/2022 3:40:46 PM	ad8e2fbb...
#7: OscillatingDropletPaper	OD3D_m3_Oh01_eta015_stagnantInit_restart3	3/23/2023 2:36:57 PM	d0b3ed73...
#8: OscillatingDropletPaper	OD3D_m3_Oh01_eta03_stagnantInit_restart2	7/11/2022 1:39:12 PM	95fa458f...
#9: OscillatingDropletPaper	OD3D_m3_Oh01_eta04_stagnantInit_restart2	7/11/2

In [27]:
NUnit.Framework.Assert.AreEqual(18, evalSessOrigin.Count, $"Not enough sessions for evaluation.");

In [28]:
List<ISessionInfo> evalSess = new List<ISessionInfo>();

In [29]:
for (int i = 0; i < mOhS.Length; i++) {
foreach (string etaCase in studyCases[i]) {

    string studyName = $"OD3D_{mOhS[i]}_Oh01_eta{etaCase}_stagnantInit";
    Console.WriteLine("Searching for: " + studyName);

    var studySess = sessions.Where(sess => sess.Name.Contains(studyName)); // && !sess.Name.Contains("Rerun"));

    int studyCount = studySess.Count();
    if (studyCount == 0) {
        //Console.WriteLine("Not found in wmg! study will be excluded from check against origin session");

        if (mOhS[i] == "m2" && etaCase == "05") {
            studyName = $"OD3D_{mOhS[i]}_Oh01_eta{etaCase}_stagnantInit_Rerun";
        }
        
        // try to get from originDB
        var originSess = originDB.Sessions.Where(sess => sess.Name.Contains(studyName));
        int originCount = originSess.Count();
        if (originCount == 0) {
            Console.WriteLine("Not found!");
        } else if (originCount > 1) {
            Console.WriteLine("Restart session found in originDB! Take last run");       
            evalSess.Add(originSess.Where(sess => sess.Name.EndsWith($"_restart{originCount-1}")).Single());
        } else {
            evalSess.Add(originSess.Single());
            Console.WriteLine("found in originDB");
        }
        
    } else if (studyCount > 1) {
        Console.WriteLine("Restart session found! Take last run");       
        evalSess.Add(studySess.Where(sess => sess.Name.EndsWith($"_restart{studyCount-1}")).Single());
    } else {
        evalSess.Add(studySess.Single());
        Console.WriteLine("found");
    }

}
}
evalSess

Searching for: OD3D_m2_Oh01_eta01_stagnantInit
found
Searching for: OD3D_m2_Oh01_eta02_stagnantInit
found
Searching for: OD3D_m2_Oh01_eta03_stagnantInit
found
Searching for: OD3D_m2_Oh01_eta04_stagnantInit
found
Searching for: OD3D_m2_Oh01_eta05_stagnantInit
found
Searching for: OD3D_m2_Oh01_eta06_stagnantInit
found
Searching for: OD3D_m2_Oh01_eta07_stagnantInit
found
Searching for: OD3D_m3_Oh01_eta015_stagnantInit
found
Searching for: OD3D_m3_Oh01_eta03_stagnantInit
found
Searching for: OD3D_m3_Oh01_eta04_stagnantInit
found
Searching for: OD3D_m3_Oh01_eta05_stagnantInit
found
Searching for: OD3D_m4_Oh01_eta01_stagnantInit
found
Searching for: OD3D_m4_Oh01_eta02_stagnantInit
found
Searching for: OD3D_m4_Oh01_eta03_stagnantInit
found
Searching for: OD3D_m4_Oh01_eta04_stagnantInit
found
Searching for: OD3D_m4_Oh01_eta05_stagnantInit
found
Searching for: OD3D_m4_Oh01_eta06_stagnantInit
found
Searching for: OD3D_m4_Oh01_eta07_stagnantInit
found


#0: OscillatingDropletPaper	OD3D_m2_Oh01_eta01_stagnantInit	12/4/2025 3:10:23 PM	90609365...
#1: OscillatingDropletPaper	OD3D_m2_Oh01_eta02_stagnantInit	12/4/2025 3:10:49 PM	0e7cdee1...
#2: OscillatingDropletPaper	OD3D_m2_Oh01_eta03_stagnantInit	12/6/2025 10:30:22 PM	42c06632...
#3: OscillatingDropletPaper	OD3D_m2_Oh01_eta04_stagnantInit	12/4/2025 3:11:22 PM	012d30cd...
#4: OscillatingDropletPaper	OD3D_m2_Oh01_eta05_stagnantInit	12/6/2025 10:30:33 PM	abb9c251...
#5: OscillatingDropletPaper	OD3D_m2_Oh01_eta06_stagnantInit	12/6/2025 10:30:45 PM	10651a2d...
#6: OscillatingDropletPaper	OD3D_m2_Oh01_eta07_stagnantInit	12/6/2025 10:30:57 PM	9b83588b...
#7: OscillatingDropletPaper	OD3D_m3_Oh01_eta015_stagnantInit	12/4/2025 3:11:58 PM	36400c0b...
#8: OscillatingDropletPaper	OD3D_m3_Oh01_eta03_stagnantInit	12/6/2025 10:31:10 PM	ca424e65...
#9: OscillatingDropletPaper	OD3D_m3_Oh01_eta04_stagnantInit	12/4/2025 3:12:10 PM	87476317...
#10: OscillatingDropletPaper	OD3D_m3_Oh01_eta05_stagnantInit*	12

In [30]:
NUnit.Framework.Assert.AreEqual(18, evalSess.Count, $"Not enough sessions for evaluation.");

### compute times

In [31]:
foreach (var sess in evalSess) {
    var nameData = sess.Name.Split(new string[] { "_" }, StringSplitOptions.RemoveEmptyEntries);
    string sessName = "";
    int nameLength = 4;
    for (int i = 0; i < nameLength; i++) {
        sessName += nameData[i];
        if (i != nameLength - 1)
            sessName += "_";
    }

    // determine compute time
    Stack<ISessionInfo>  procSIs = new Stack<ISessionInfo>();
    procSIs.Push(sess);
    var currSI = sess;
    int timesteps = currSI.Timesteps.Last().TimeStepNumber.MajorNumber;
    double physicalTime = currSI.Timesteps.Last().PhysicalTime;
    TimeSpan computeTime = currSI.Timesteps.Last().CreationTime - currSI.Timesteps.First().CreationTime;
    var rSIs = sessions.Where(sess => sess.ID.Equals(currSI.RestartedFrom));
    while(!rSIs.IsNullOrEmpty()) {
        var rSI = rSIs.Single();
        procSIs.Push(rSI);
        currSI = rSI;
        computeTime = computeTime + (currSI.Timesteps.Last().CreationTime - currSI.Timesteps.First().CreationTime);
        rSIs = sessions.Where(sess => sess.ID.Equals(currSI.RestartedFrom));
    }

    Console.WriteLine($"Session - {sessName}: timesteps = {timesteps} (t_end = {physicalTime}); compute time = {computeTime.Days}:{computeTime.Hours}");
}

Session - OD3D_m2_Oh01_eta01: timesteps = 9 (t_end = 0.045); compute time = 0:5
Session - OD3D_m2_Oh01_eta02: timesteps = 8 (t_end = 0.04); compute time = 0:4
Session - OD3D_m2_Oh01_eta03: timesteps = 169 (t_end = 0.8450000000000006); compute time = 3:20
Session - OD3D_m2_Oh01_eta04: timesteps = 8 (t_end = 0.04); compute time = 0:4
Session - OD3D_m2_Oh01_eta05: timesteps = 200 (t_end = 1.0000000000000007); compute time = 4:19
Session - OD3D_m2_Oh01_eta06: timesteps = 199 (t_end = 0.9950000000000008); compute time = 4:21
Session - OD3D_m2_Oh01_eta07: timesteps = 199 (t_end = 0.9950000000000008); compute time = 4:22
Session - OD3D_m3_Oh01_eta015: timesteps = 9 (t_end = 0.045); compute time = 0:4
Session - OD3D_m3_Oh01_eta03: timesteps = 139 (t_end = 0.6950000000000005); compute time = 3:13
Session - OD3D_m3_Oh01_eta04: timesteps = 10 (t_end = 0.049999999999999996); compute time = 0:6
Session - OD3D_m3_Oh01_eta05: timesteps = 1 (t_end = 0); compute time = 0:7
Session - OD3D_m4_Oh01_eta01:

### read log data

In [32]:
public static List<Plot2Ddata> ReadLogDataForSphericalHarmonics(this List<ISessionInfo> sessions) {

    List<Plot2Ddata> plotData = new List<Plot2Ddata>();

    int numberSessions = sessions.Count();

    // Read all data
    for(int j = 0; j < numberSessions; j++) {

        var sess = sessions.ElementAt(j);
        Console.WriteLine("Processing session: " + sess.Name);

        // Get session history
        List<Guid> restartSessionList = new List<Guid>();
        restartSessionList.Add(sess.ID);
        Guid restartID = sess.RestartedFrom;

        var sessionIDs = new List<Guid>();
        var directories = Directory.GetDirectories(Path.Combine(sess.Database.Path, "sessions"));
        foreach(string dirname in directories) {
            string IDname = dirname.Split(new string[] { "\\" }, StringSplitOptions.RemoveEmptyEntries).Last();
            sessionIDs.Add(new Guid(IDname));
        }

        while(restartID != Guid.Empty) {
            try {
                var rSess = sessionIDs.Where(sess => sess == restartID).Single();
                Console.WriteLine("  Found restart session: " + rSess);
                restartSessionList.Add(rSess);

                string path_obj = Path.Combine(sess.Database.Path, "sessions", restartID.ToString(), "Control-obj.txt");
                string ctrlfileContent = File.ReadAllText(path_obj);
                var ctrl = (XNSE_Control)AppControl.Deserialize(ctrlfileContent);

                if(ctrl.RestartInfo != null) {
                    restartID = ctrl.RestartInfo.Item1;
                } else {
                    restartID = Guid.Empty;
                }

            } catch { // do nothing if session not found
                restartID = Guid.Empty;
            }
        }
        restartSessionList.Reverse();

        Console.WriteLine("  Merging data from " + restartSessionList.Count + " sessions.");

        string path = Path.Combine(sess.Database.Path, "sessions", sess.ID.ToString(), "SphericalHarmonics_NEW.txt");   // there might be some updated data in the original files
        string[] lines;
        try {
            lines = File.ReadAllLines(path);
        } catch {
            path = Path.Combine(sess.Database.Path, "sessions", sess.ID.ToString(), "SphericalHarmonics.txt");
            lines = File.ReadAllLines(path);
        }
        string[] modes = lines[0].Split(new string[] { "\t" }, StringSplitOptions.RemoveEmptyEntries).Skip(1).ToArray();

        Console.WriteLine("  Found modes: " + string.Join(", ", modes));

        List<double> time = new List<double>();
        int numModes = modes.Length;
        List<double>[] modesData = new List<double>[numModes];
        for(int m = 0; m < numModes; m++) {
            modesData[m] = new List<double>();
        }

        for(int rSi = 0; rSi < restartSessionList.Count(); rSi++) {

            path = Path.Combine(sess.Database.Path, "sessions", restartSessionList.ElementAt(rSi).ToString(), "SphericalHarmonics_NEW.txt");
            try {
                lines = File.ReadAllLines(path);
            } catch {
                path = Path.Combine(sess.Database.Path, "sessions", restartSessionList.ElementAt(rSi).ToString(), "SphericalHarmonics.txt");
                lines = File.ReadAllLines(path);
            }
            lines = File.ReadAllLines(path);

            // skip SetUpData
            int line0 = 0;
            for(int i = 0; i < lines.Length; i++) {
                string check = lines[i].Split(new string[] { "\t" }, StringSplitOptions.RemoveEmptyEntries)[0];
                if(check.Equals("time") || check.Contains("#")) {
                    line0 = i + 1;
                    break;
                }
            }

            for(int i = 0; i < lines.Length - line0; i++) {

                double l_time = Convert.ToDouble(lines[line0 + i].Split(new string[] { "\t" }, StringSplitOptions.RemoveEmptyEntries)[0]);
                double[] l_values = new double[numModes];
                for(int m = 0; m < numModes; m++) {
                    l_values[m] = Convert.ToDouble(lines[line0 + i].Split(new string[] { "\t" }, StringSplitOptions.RemoveEmptyEntries)[m + 1]);
                }

                if(time.IsNullOrEmpty() || l_time > time.Last()) {
                    time.Add(l_time);
                    for(int m = 0; m < numModes; m++) {
                        modesData[m].Add(l_values[m]);
                    }
                }
            }

        }

        // Build DataSets
        Console.WriteLine("Build DataSets");
    
        KeyValuePair<string, double[][]>[] dataRowsValue = new KeyValuePair<string, double[][]>[numModes];
        for(int m = 0; m < numModes; m++) {
            dataRowsValue[m] = new KeyValuePair<string, double[][]>(modes[m], new double[][] { time.ToArray(), modesData[m].ToArray() });
        }          
        Plot2Ddata plt = new Plot2Ddata(dataRowsValue);
        plt.Title = sess.Name.Replace("_", "-");
        plt.Xlabel = "time";
        plt.Ylabel = "a_m";
        
        plotData.Add(plt);      

        Console.WriteLine("... Done"); 

    }

    return plotData;
}

### Aspect ratios

In [33]:
var modeData = evalSess.ReadLogDataForSphericalHarmonics();

Processing session: OD3D_m2_Oh01_eta01_stagnantInit
  Merging data from 1 sessions.
  Found modes: (0, 0), (1, 0), (2, 0), (3, 0), (4, 0), (5, 0), (6, 0), (7, 0), (8, 0)
Build DataSets
... Done
Processing session: OD3D_m2_Oh01_eta02_stagnantInit
  Merging data from 1 sessions.
  Found modes: (0, 0), (1, 0), (2, 0), (3, 0), (4, 0), (5, 0), (6, 0), (7, 0), (8, 0)
Build DataSets
... Done
Processing session: OD3D_m2_Oh01_eta03_stagnantInit
  Merging data from 1 sessions.
  Found modes: (0, 0), (1, 0), (2, 0), (3, 0), (4, 0), (5, 0), (6, 0), (7, 0), (8, 0), (9, 0), (10, 0), (11, 0), (12, 0)
Build DataSets
... Done
Processing session: OD3D_m2_Oh01_eta04_stagnantInit
  Merging data from 1 sessions.
  Found modes: (0, 0), (1, 0), (2, 0), (3, 0), (4, 0), (5, 0), (6, 0), (7, 0), (8, 0)
Build DataSets
... Done
Processing session: OD3D_m2_Oh01_eta05_stagnantInit
  Merging data from 1 sessions.
  Found modes: (0, 0), (1, 0), (2, 0), (3, 0), (4, 0), (5, 0), (6, 0), (7, 0), (8, 0), (9, 0), (10, 0), (

In [34]:
var modeDataOrigin = evalSessOrigin.ReadLogDataForSphericalHarmonics();

Processing session: OD3D_m2_Oh01_eta01_stagnantInit_restart3
  Found restart session: a2d5a18a-39f4-449a-8013-54106406c1cf
  Found restart session: 2a51c1d4-cf2c-45f6-9baf-be68918ddc54
  Found restart session: 59bb73dc-0683-42a1-a615-e87ed3524000
  Merging data from 4 sessions.
  Found modes: (0, 0), (1, 0), (2, 0), (3, 0), (4, 0), (5, 0), (6, 0), (7, 0), (8, 0)
Build DataSets
... Done
Processing session: OD3D_m2_Oh01_eta02_stagnantInit_restart3
  Found restart session: 33d67916-8020-4464-a16e-f55562a81bb0
  Found restart session: 6b199ada-b490-49b1-99d1-0a7ec09620fd
  Found restart session: 2e275baf-7a62-4b31-a3af-e15cf125f40c
  Merging data from 4 sessions.
  Found modes: (0,0), (1,0), (2,0), (3,0), (4,0), (5,0), (6,0), (7,0), (8,0), (9,0), (10,0), (11,0), (12,0)
Build DataSets
... Done
Processing session: OD3D_m2_Oh01_eta03_stagnantInit_restart1
  Found restart session: 56519a35-fccc-4934-9bb1-f69e5a25a481
  Merging data from 2 sessions.
  Found modes: (0, 0), (1, 0), (2, 0), (3, 0)

In [35]:
public static double[] GetAspectRatioByModeDecomposition(Plot2Ddata modeData, int reqMaxL = -1) {

    int numTimesteps = modeData.dataGroups[0].Abscissas.Length;
    double[] aspectRatio = new double[numTimesteps];

    for (int ts = 0; ts < numTimesteps; ts++) {

        double northPole = 0.0;
        double southPole = 0.0;
        double equatorWidth = 0.0;

        int maxL = modeData.dataGroups.Count();
        if (reqMaxL >= 0) {
            maxL = Math.Min(maxL, reqMaxL);
        }
        for (int m = 0; m < maxL; m++) {

            double mValue = modeData.dataGroups[m].Values[ts];

            northPole += mValue * SphericalHarmonics.MyLegendre(m,0,Math.Cos(0.0));
            southPole += mValue * SphericalHarmonics.MyLegendre(m,0,Math.Cos(Math.PI));
            equatorWidth += mValue * SphericalHarmonics.MyLegendre(m,0,Math.Cos(Math.PI/2.0));   
        }
        aspectRatio[ts] = (northPole + southPole)/(2.0 * equatorWidth);      
    }

    return aspectRatio;
}

In [36]:
public static Plot2Ddata GetAspectRatioStudyOverTimeByModeDecomposition_Plot2Ddata(List<Plot2Ddata> modeDataS, string[] studies, string[] legendNames = null, int maxL = -1) {

    Plot2Ddata plt =  new Plot2Ddata();
    plt.Xlabel = "t";
    plt.Ylabel = "L/W";

    int lcIdx = 0;
    for (int i = 0; i < studies.Length; i++) {
        var datPlt = modeDataS.Where(plt => plt.Title.Contains(studies[i])).Single();

        double[] aspectRatio = GetAspectRatioByModeDecomposition(datPlt, maxL);
    
        var fmt = new PlotFormat();
        fmt.Style = Styles.Lines;
        fmt.DashType = DashTypes.Solid;
        fmt.LineWidth = 2;
        //LineColors[] customColors = new LineColors[] { LineColors.Blue, LineColors.Red, LineColors.Green };
        fmt.LineColor = (LineColors)(lcIdx);

        string[] nameParts = datPlt.Title.Split(new string[] { "-" }, StringSplitOptions.RemoveEmptyEntries);
        string name = legendNames != null ? legendNames[i] : nameParts[3];
        plt.AddDataGroup($"{name}", datPlt.dataGroups[0].Abscissas, aspectRatio, fmt);

        lcIdx++;      
    }   

    plt.XrangeMin = 0.0;
    plt.XrangeMax = 4.0;

    plt.ShowLegend = true;

    return plt;        
}

### FIG. 17.
Aspect ratio $L/W$ over time $t$ for drops with $Oh = 0.1$ in the different modes of initial deformation
(a) $m = 2$, (b) $m = 3$, and (c) $m = 4$, with deformation parameters up to $ \eta_0 = 0.7$. The black dashed lines
indicate WNLT results.

In [37]:
List<Plot2Ddata> plotModeData = (runShortTests) ? modeDataOrigin : modeData;

In [ ]:
public static MultidimensionalArray ReadPaperData(string dat) {

    string userName = System.Security.Principal.WindowsIdentity.GetCurrent().Name;
    string[] nameData = dat.Split(new string[] { "_" }, StringSplitOptions.RemoveEmptyEntries);

    string dataPath = (userName.Equals(@"FDY\jenkinsci")) ? dat : $"../paperData/AspectRatioOverTime/" + nameData[1] + "/" + dat;

    //return IMatrixExtensions.LoadFromTextFile(dataPath);
    int headerSize = 2;
    using(StreamReader txt = new StreamReader(dataPath)) {
        try {
            for (int i = 0; i < headerSize; i++) {
                txt.ReadLine(); // skip header
            }
            return IMatrixExtensions.LoadFromStream(txt);
        }   finally {
            txt.Close();
        }
    }
}

In [ ]:
Plot2Ddata[,] Fig17 = new Plot2Ddata[2, 2];

//string userName = System.Security.Principal.WindowsIdentity.GetCurrent().Name;
var fmt = new PlotFormat();
fmt.Style = Styles.Lines; 
fmt.LineWidth = 2;
fmt.LineColor = LineColors.Black;


//var fig = GetAspectRatioOverTime_Plot2Ddata(evalStudyData, new string[] { "m2-Oh01-eta01-stagnantInit", "m2-Oh01-eta01-thirdOrderInit" });
var fig = GetAspectRatioStudyOverTimeByModeDecomposition_Plot2Ddata(plotModeData, 
            new string[] { "m2-Oh01-eta01-stagnantInit", "m2-Oh01-eta02-stagnantInit", "m2-Oh01-eta03-stagnantInit", "m2-Oh01-eta04-stagnantInit",
                            "m2-Oh01-eta05-stagnantInit", "m2-Oh01-eta06-stagnantInit", "m2-Oh01-eta07-stagnantInit" },
            new string[] { "eta_0 = 0.1", "eta_0 = 0.2", "eta_0 = 0.3", "eta_0 = 0.4", "eta_0 = 0.5", "eta_0 = 0.6", "eta_0 = 0.7" });
var wnlt_dat = ReadPaperData("m2_Oh01_eta01_WNLT.txt");
fig.AddDataGroup("eta_0 = 0.1", wnlt_dat.GetColumn(0), wnlt_dat.GetColumn(1), fmt);
wnlt_dat = ReadPaperData("m2_Oh01_eta02_WNLT.txt");
fig.AddDataGroup("eta_0 = 0.2", wnlt_dat.GetColumn(0), wnlt_dat.GetColumn(1), fmt);
fig.YrangeMin = 0.5;
fig.YrangeMax = 3.0;
Fig17[0, 0] = fig; 

//var fig = GetAspectRatioOverTime_Plot2Ddata(evalStudyData, new string[] { "m2-Oh01-eta01-stagnantInit", "m2-Oh01-eta01-thirdOrderInit" });
fig = GetAspectRatioStudyOverTimeByModeDecomposition_Plot2Ddata(plotModeData, 
            new string[] { "m3-Oh01-eta015-stagnantInit", "m3-Oh01-eta03-stagnantInit", "m3-Oh01-eta04-stagnantInit", "m3-Oh01-eta05-stagnantInit" },
            new string[] { "eta_0 = 0.15", "eta_0 = 0.3", "eta_0 = 0.4", "eta_0 = 0.5" },
            20);
wnlt_dat = ReadPaperData("m3_Oh01_eta015_WNLT.txt");
fig.AddDataGroup("eta_0 = 0.15", wnlt_dat.GetColumn(0), wnlt_dat.GetColumn(1), fmt);
fig.YrangeMin = 0.9;
fig.YrangeMax = 1.25;
Fig17[0, 1] = fig;

//var fig = GetAspectRatioOverTime_Plot2Ddata(evalStudyData, new string[] { "m2-Oh01-eta01-stagnantInit", "m2-Oh01-eta01-thirdOrderInit" });
fig = GetAspectRatioStudyOverTimeByModeDecomposition_Plot2Ddata(plotModeData, 
            new string[] { "m4-Oh01-eta01-stagnantInit", "m4-Oh01-eta02-stagnantInit", "m4-Oh01-eta03-stagnantInit", "m4-Oh01-eta04-stagnantInit",
                            "m4-Oh01-eta05-stagnantInit", "m4-Oh01-eta06-stagnantInit", "m4-Oh01-eta07-stagnantInit" },
            new string[] { "eta_0 = 0.1", "eta_0 = 0.2", "eta_0 = 0.3", "eta_0 = 0.4", "eta_0 = 0.5", "eta_0 = 0.6", "eta_0 = 0.7" });
wnlt_dat = ReadPaperData("m4_Oh01_eta01_WNLT.txt");
fig.AddDataGroup("eta_0 = 0.1", wnlt_dat.GetColumn(0), wnlt_dat.GetColumn(1), fmt);
wnlt_dat = ReadPaperData("m4_Oh01_eta03_WNLT.txt");
fig.AddDataGroup("eta_0 = 0.3", wnlt_dat.GetColumn(0), wnlt_dat.GetColumn(1), fmt);
fig.YrangeMin = 0.8;
fig.YrangeMax = 1.6;
Fig17[1, 0] = fig;

Fig17.ToGnuplot().PlotSVG(xRes:1200,yRes:1000)

Using gnuplot: C:\Program Files (x86)\FDY\BoSSS\bin\native\win\gnuplot-gp510-20160418-win32-mingw\gnuplot\bin\gnuplot.exe
Note: In a Jupyter Worksheet, you must NOT have a trailing semicolon in order to see the plot on screen; otherwise, the output migth be surpressed.!


<?xml version="1.0" encoding="utf-8" standalone="no"?>
<!DOCTYPE svg PUBLIC "-//W3C//DTD SVG 1.1//EN"
 "http://www.w3.org/Graphics/SVG/1.1/DTD/svg11.dtd">
 

 Gnuplot 
 Produced by GNUPLOT 5.1 patchlevel 0 

 

 
 

 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 0.5 
 
 
 
 
 1 
 
 
 
 
 1.5 
 
 
 
 
 2 
 
 
 
 
 2.5 
 
 
 
 
 3 
 
 
 
 
 0 
 
 
 
 
 0.5 
 
 
 
 
 1 
 
 
 
 
 1.5 
 
 
 
 
 2 
 
 
 
 
 2.5 
 
 
 
 
 3 
 
 
 
 
 3.5 
 
 
 
 
 4 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 L/W 
 
 
 
 
 t 
 
 
 
 
 eta0 = 0.1 
 
 
 
 
 eta 0 = 0.1 
 
 
 
	<path stroke='gray' d='M371.4,62.1 L424.8,62.1 M77.8,333.1 L78.4,333.1 L79.0,333.1 L79.5,333.1 L80.1,333.2 L80.7,333.2
 L81.3,333.2 L81.9,333.2 L82.5,333.3 L83.0,333.3 L83.6,333.3 L84.2,333.4 L84.8,333.4 L85.4,333.5
 L86.0,333.5 L86.5,333.6 L87.1,333.7 L87.7,333.7 L88.3,333.8 L88.9,333.9 L89.5,333.9 L90.0,334.0
 L90.6,334.1 L91.2,334.2 L91.8,334.3 L92.4,334.4 L93.0,334.5 L93.5,334.6 L94.1,334.7 L94.7,334.8
 L95.3,334.9 L95.9,335.1 L96.5,335.2 L97.0,335.3 L97.6,335.4 L98.2,335.6 L98.8,335.7 L99.4,335.9
 L99.9,336.0 L100.5,336.1 L101.1,336.3 L101.7,336.5 L102.3,336.6 L102.9,336.8 L103.4,336.9 L104.0,337.1
 L104.6,337.3 L105.2,337.4 L105.8,337.6 L106.4,337.8 L106.9,338.0 L107.5,338.1 L108.1,338.3 L108.7,338.5
 L109.3,338.7 L109.9,338.9 L110.4,339.1 L111.0,339.3 L111.6,339.5 L112.2,339.7 L112.8,339.9 L113.4,340.1
 L113.9,340.3 L114.5,340.6 L115.1,340.8 L115.7,341.0 L116.3,341.2 L116.9,341.4 L117.4,341.7 L118.0,341.9
 L118.6,342.1 L119.2,342.4 L119.8,342.6 L120.3,342.8 L120.9,343.1 L121.5,343.3 L122.1,343.6 L122.7,343.8
 L123.3,344.1 L123.8,344.3 L124.4,344.6 L125.0,344.8 L125.6,345.1 L126.2,345.3 L126.8,345.6 L127.3,345.8
 L127.9,346.1 L128.5,346.4 L129.1,346.6 L129.7,346.9 L130.3,347.2 L130.8,347.4 L131.4,347.7 L132.0,348.0
 L132.6,348.2 L133.2,348.5 L133.8,348.8 L134.3,349.1 L134.9,349.3 L135.5,349.6 L136.1,349.9 L136.7,350.2
 L137.3,350.4 L137.8,350.7 L138.4,351.0 L139.0,351.3 L139.6,351.5 L140.2,351.8 L140.8,352.1 L141.3,352.4
 L141.9,352.6 L142.5,352.9 L143.1,353.2 L143.7,353.5 L144.2,353.8 L144.8,354.0 L145.4,354.3 L146.0,354.6
 L146.6,354.9 L147.2,355.1 L147.7,355.4 L148.3,355.7 L148.9,355.9 L149.5,356.2 L150.1,356.5 L150.7,356.8
 L151.2,357.0 L151.8,357.3 L152.4,357.6 L153.0,357.8 L153.6,358.1 L154.2,358.3 L154.7,358.6 L155.3,358.9
 L155.9,359.1 L156.5,359.4 L157.1,359.6 L157.7,359.9 L158.2,360.1 L158.8,360.4 L159.4,360.6 L160.0,360.9
 L160.6,361.1 L161.2,361.4 L161.7,361.6 L162.3,361.8 L162.9,362.1 L163.5,362.3 L164.1,362.5 L164.6,362.8
 L165.2,363.0 L165.8,363.2 L166.4,363.5 L167.0,363.7 L167.6,363.9 L168.1,364.1 L168.7,364.3 L169.3,364.5
 L169.9,364.8 L170.5,365.0 L171.1,365.2 L171.6,365.4 L172.2,365.6 L172.8,365.8 L173.4,366.0 L174.0,366.2
 L174.6,366.3 L175.1,366.5 L175.7,366.7 L176.3,366.9 L176.9,367.1 L177.5,367.3 L178.1,367.4 L178.6,367.6
 L179.2,367.8 L179.8,367.9 L180.4,368.1 L181.0,368.3 L181.6,368.4 L182.1,368.6 L182.7,368.7 L183.3,368.9
 L183.9,369.0 L184.5,369.1 L185.0,369.3 L185.6,369.4 L186.2,369.6 L186.8,369.7 L187.4,369.8 L188.0,369.9
 L188.5,370.1 L189.1,370.2 L189.7,370.3 L190.3,370.4 L190.9,370.5 L191.5,370.6 L192.0,370.7 L192.6,370.8
 L193.2,370.9 L193.8,371.0 L194.4,371.1 L195.0,371.2 L195.5,371.3 L196.1,371.4 L196.7,371.5 L197.3,371.5
 L197.9,371.6 L198.5,371.7 L199.0,371.7 L199.6,371.8 L200.2,371.9 L200.8,371.9 L201.4,372.0 L202.0,372.0
 L202.5,372.1 L203.1,372.1 L203.7,372.2 L204.3,372.2 L204.9,372.3 L205.4,372.3 L206.0,372.3 L206.6,372.4
 L207.2,372.4 L207.8,372.4 L208.4,372.4 L208.9,372.4 L209.5,372.5 L210.1,372.5 L210.7,372.5 L211.3,372.5
 L211.9,372.5 L212.4,372.5 L213.0,372.5 L213.6,372.5 L214.2,372.5 L214.8,372.5 L215.4,372.5 L215.9,372.4
 L216.5,372.4 L217.1,372.4 L217.7,372.4 L218.3,372.4 L218.9,372.3 L219.4,372.3 L220.0,372.3 L220.6,372.2
 L221.2,372.2 L221.8,372.1 L222.4,372.1 L222.9,372.0 L223.5,372.0 L224

In [40]:
public static void CheckAgainstOriginModeData(List<Plot2Ddata> StudyData, List<Plot2Ddata> OriginData, string mStudy, double errThrsh, int maxL = 6, bool checkOddModes = false, bool output = false) {

    foreach (var study in StudyData) {

        if (!study.Title.Contains(mStudy)) {
            continue;
        }

        //var Splt = StudyData.Where(grp => grp.Title.Contains(study)).Single();
        var Oplt = OriginData.Where(grp => grp.Title.Contains(study.Title)).Single();

        Console.WriteLine($"checking session {study.Title}");

        //int mode = 0;
        foreach (var Sgrp in study.dataGroups) {    

            int mode = Int32.Parse(string.Concat(Sgrp.Name.Where( Char.IsDigit ).SkipLast(1)));
            
            if (mode > maxL)
                continue;

            if (!checkOddModes && mode % 2 == 1) {
                mode++;
                continue;  
            }

            var Ogrp = Oplt.dataGroups.Where(grp => Int32.Parse(string.Concat(grp.Name.Where( Char.IsDigit ).SkipLast(1))) == mode).Single();
            if (Sgrp.Abscissas.Length == 0 || Ogrp.Abscissas.Length == 0) {
                Console.WriteLine("no data available - skipping mode ...");
                continue;
            }
            var errorData = ISessionInfoExtensions.ComputeErrorNormsForLogData(Sgrp, Ogrp);

            if (output) {
                Console.WriteLine($"checking mode {mode}:"); 

                Console.WriteLine($"    l1-error norm = {errorData["l_1 error norm"]}");
                Console.WriteLine($"    l2-error norm = {errorData["l_2 error norm"]}");
            }
        
            NUnit.Framework.Assert.LessOrEqual(errorData["l_1 error norm"], errThrsh,  
                $"l_1 error norm larger than allowed: {errorData["l_1 error norm"]}");

            NUnit.Framework.Assert.LessOrEqual(errorData["l_2 error norm"], errThrsh,  
                $"l_2 error norm larger than allowed: {errorData["l_2 error norm"]}");

            mode++;
        }
    }

}

In [41]:
CheckAgainstOriginModeData(modeData, modeDataOrigin, "m2", 0.008, 6, false, false) 

checking session OD3D-m2-Oh01-eta01-stagnantInit
checking session OD3D-m2-Oh01-eta02-stagnantInit
checking session OD3D-m2-Oh01-eta03-stagnantInit
checking session OD3D-m2-Oh01-eta04-stagnantInit
checking session OD3D-m2-Oh01-eta05-stagnantInit
checking session OD3D-m2-Oh01-eta06-stagnantInit
checking session OD3D-m2-Oh01-eta07-stagnantInit


In [42]:
CheckAgainstOriginModeData(modeData, modeDataOrigin, "m3", 0.8, 6, true, false) 

checking session OD3D-m3-Oh01-eta015-stagnantInit
checking session OD3D-m3-Oh01-eta03-stagnantInit
checking session OD3D-m3-Oh01-eta04-stagnantInit
checking session OD3D-m3-Oh01-eta05-stagnantInit
no data available - skipping mode ...
no data available - skipping mode ...
no data available - skipping mode ...
no data available - skipping mode ...
no data available - skipping mode ...
no data available - skipping mode ...
no data available - skipping mode ...


In [43]:
CheckAgainstOriginModeData(modeData, modeDataOrigin, "m4", 0.04, 6, false, false) 

checking session OD3D-m4-Oh01-eta01-stagnantInit
checking session OD3D-m4-Oh01-eta02-stagnantInit
checking session OD3D-m4-Oh01-eta03-stagnantInit
checking session OD3D-m4-Oh01-eta04-stagnantInit
checking session OD3D-m4-Oh01-eta05-stagnantInit
checking session OD3D-m4-Oh01-eta06-stagnantInit
checking session OD3D-m4-Oh01-eta07-stagnantInit


### Energies (kinetic, surface, total)

In [47]:
public static Plot2Ddata GetEnergy_Plot2Ddata(List<ISessionInfo> studySess, string valueName, string[] studies, double E0 = 0.0, string dataName = null, string[] legendNames = null) {

    //var dataGrps = data.ElementAt(dataIndex).dataGroups;

    Plot2Ddata plt =  new Plot2Ddata();
    plt.Xlabel = "Time";
    plt.Ylabel = valueName;

    int lcIdx = 1;
    for (int i = 0; i < studies.Length; i++) {

        List<ISessionInfo> dataSess = studySess.Where(sess => sess.Name.Contains(studies[i])).ToList<ISessionInfo>();
        // Console.WriteLine($"{study}:");
        // Console.WriteLine($"    number sessions: {data.Count()}");
        if (dataSess.Count() != 1) 
            throw new ArgumentException();

        Plot2Ddata data;
        try {
            data = dataSess.ReadLogDataValue( "Energy", valueName);
        } catch {
            data = dataSess.ReadLogDataValue( "EnergyLogValues", valueName);
        }
        
        var datGrp = data.dataGroups[0];

        double[] energyValues = datGrp.Values.CloneAs();
        for (int j = 0; j < energyValues.Length; j++) { // multiply by 4.0 to get the energy of the whole droplet (comp domain only a quarter)
            if (E0 != 0.0) {
                // shift energy by E0
                energyValues[j] = (4.0 * energyValues[j]) - E0; 
            } else 
                energyValues[j] = (4.0 * energyValues[j]);
        }


        var fmt = new PlotFormat();
        fmt.Style = Styles.Lines; 
        fmt.DashType = DashTypes.Solid;
        fmt.LineWidth = 2;
        fmt.LineColor = (LineColors)(lcIdx); //LineColors.Blue;

        string[] nameParts = datGrp.Name.Split(new string[] { "-" }, StringSplitOptions.RemoveEmptyEntries);
        string name = legendNames != null ? legendNames[i] : nameParts[3];
        plt.AddDataGroup($"{name}", datGrp.Abscissas, energyValues, fmt);

        lcIdx++;
    }

    plt.XrangeMin = 0.0;
    plt.XrangeMax = 4.0;
    plt.ShowLegend = true;

    return plt;

} 

### Fig. 21 
Kinetic bulk energy $E_{kin} = \frac{1}{2} \int_{\mathfrak{A}} \vec{u} \cdot \vec{u} dV$ over time $t$ for the different modes of initial deformation
(a) $m = 2$, (b) $m = 3$, and (c) $m = 4$, with deformation parameters up to $\eta_0 = 0.7$ and Ohnesorge number
$Oh = 0.1$. The red dash-dotted line indicates WNLT results.


In [48]:
List<ISessionInfo> plotEvalSess = (runShortTests) ? evalSessOrigin : evalSess;

In [49]:
Plot2Ddata[,] Fig21 = new Plot2Ddata[2, 2];

//var fig = GetAspectRatioOverTime_Plot2Ddata(evalStudyData, new string[] { "m2-Oh01-eta01-stagnantInit", "m2-Oh01-eta01-thirdOrderInit" });
var fig = GetEnergy_Plot2Ddata(plotEvalSess, "kineticEnergy",
            new string[] { "m2_Oh01_eta01_stagnantInit", "m2_Oh01_eta02_stagnantInit", "m2_Oh01_eta03_stagnantInit", "m2_Oh01_eta04_stagnantInit",
                            "m2_Oh01_eta05_stagnantInit", "m2_Oh01_eta06_stagnantInit", "m2_Oh01_eta07_stagnantInit" },
            legendNames: new string[] { "eta_0 = 0.1", "eta_0 = 0.2", "eta_0 = 0.3", "eta_0 = 0.4", "eta_0 = 0.5", "eta_0 = 0.6", "eta_0 = 0.7" });
fig.YrangeMin = 0.0;
fig.YrangeMax = 1.0;
Fig21[0, 0] = fig; 

//var fig = GetAspectRatioOverTime_Plot2Ddata(evalStudyData, new string[] { "m2-Oh01-eta01-stagnantInit", "m2-Oh01-eta01-thirdOrderInit" });
fig = GetEnergy_Plot2Ddata(plotEvalSess, "kineticEnergy",
            new string[] { "m3_Oh01_eta015_stagnantInit", "m3_Oh01_eta03_stagnantInit", "m3_Oh01_eta04_stagnantInit", "m3_Oh01_eta05_stagnantInit" },
            legendNames: new string[] { "eta_0 = 0.15", "eta_0 = 0.3", "eta_0 = 0.4", "eta_0 = 0.5" });
fig.YrangeMin = 0.0;
fig.YrangeMax = 1.0;
Fig21[0, 1] = fig;

//var fig = GetAspectRatioOverTime_Plot2Ddata(evalStudyData, new string[] { "m2-Oh01-eta01-stagnantInit", "m2-Oh01-eta01-thirdOrderInit" });
fig = GetEnergy_Plot2Ddata(plotEvalSess, "kineticEnergy",
            new string[] { "m4_Oh01_eta01_stagnantInit", "m4_Oh01_eta02_stagnantInit", "m4_Oh01_eta03_stagnantInit", "m4_Oh01_eta04_stagnantInit",
                            "m4_Oh01_eta05_stagnantInit", "m4_Oh01_eta06_stagnantInit", "m4_Oh01_eta07_stagnantInit" },
            legendNames: new string[] { "eta_0 = 0.1", "eta_0 = 0.2", "eta_0 = 0.3", "eta_0 = 0.4", "eta_0 = 0.5", "eta_0 = 0.6", "eta_0 = 0.7" });
fig.YrangeMin = 0.0;
fig.YrangeMax = 2.0;
Fig21[1, 0] = fig;

Fig21.ToGnuplot().PlotSVG(xRes:1200,yRes:1000)

Processing session: OD3D_m2_Oh01_eta01_stagnantInit_restart3
  Found restart session: a2d5a18a-39f4-449a-8013-54106406c1cf
  Found restart session: 2a51c1d4-cf2c-45f6-9baf-be68918ddc54
  Found restart session: 59bb73dc-0683-42a1-a615-e87ed3524000
  Merging data from 4 sessions.
Processing session: OD3D_m2_Oh01_eta01_stagnantInit_restart3
  Found restart session: a2d5a18a-39f4-449a-8013-54106406c1cf
  Found restart session: 2a51c1d4-cf2c-45f6-9baf-be68918ddc54
  Found restart session: 59bb73dc-0683-42a1-a615-e87ed3524000
  Merging data from 4 sessions.
... Done
Build DataSet
Processing session: OD3D_m2_Oh01_eta02_stagnantInit_restart3
  Found restart session: 33d67916-8020-4464-a16e-f55562a81bb0
  Found restart session: 6b199ada-b490-49b1-99d1-0a7ec09620fd
  Found restart session: 2e275baf-7a62-4b31-a3af-e15cf125f40c
  Merging data from 4 sessions.
Processing session: OD3D_m2_Oh01_eta02_stagnantInit_restart3
  Found restart session: 33d67916-8020-4464-a16e-f55562a81bb0
  Found restart s

<?xml version="1.0" encoding="utf-8" standalone="no"?>
<!DOCTYPE svg PUBLIC "-//W3C//DTD SVG 1.1//EN"
 "http://www.w3.org/Graphics/SVG/1.1/DTD/svg11.dtd">
 

 Gnuplot 
 Produced by GNUPLOT 5.1 patchlevel 0 

 

 
 

 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 0 
 
 
 
 
 0.2 
 
 
 
 
 0.4 
 
 
 
 
 0.6 
 
 
 
 
 0.8 
 
 
 
 
 1 
 
 
 
 
 0 
 
 
 
 
 0.5 
 
 
 
 
 1 
 
 
 
 
 1.5 
 
 
 
 
 2 
 
 
 
 
 2.5 
 
 
 
 
 3 
 
 
 
 
 3.5 
 
 
 
 
 4 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 kineticEnergy 
 
 
 
 
 Time 
 
 
 
 
 eta0 = 0.1 
 
 
 
 
 eta 0 = 0.1 
 
 
 
	<path stroke='rgb(255, 0, 0)' d='M371.4,62.1 L424.8,62.1 M77.8,437.5 L78.4,437.5 L79.0,437.5 L79.5,437.5 L80.1,437.4 L80.7,437.4
 L81.3,437.4 L81.9,437.3 L82.5,437.3 L83.0,437.2 L83.6,437.2 L84.2,437.1 L84.8,437.0 L85.4,437.0
 L86.0,436.9 L86.5,436.8 L87.1,436.7 L87.7,436.6 L88.3,436.5 L88.9,436.4 L89.5,436.3 L90.0,436.2
 L90.6,436.0 L91.2,435.9 L91.8,435.8 L92.4,435.6 L93.0,435.5 L93.5,435.4 L94.1,435.2 L94.7,435.1
 L95.3,434.9 L95.9,434.8 L96.5,434.6 L97.0,434.5 L97.6,434.3 L98.2,434.1 L98.8,434.0 L99.4,433.8
 L99.9,433.6 L100.5,433.5 L101.1,433.3 L101.7,433.1 L102.3,432.9 L102.9,432.8 L103.4,432.6 L104.0,432.4
 L104.6,432.2 L105.2,432.0 L105.8,431.9 L106.4,431.7 L106.9,431.5 L107.5,431.3 L108.1,431.1 L108.7,430.9
 L109.3,430.8 L109.9,430.6 L110.4,430.4 L111.0,430.2 L111.6,430.0 L112.2,429.9 L112.8,429.7 L113.4,429.5
 L113.9,429.3 L114.5,429.2 L115.1,429.0 L115.7,428.8 L116.3,428.6 L116.9,428.5 L117.4,428.3 L118.0,428.2
 L118.6,428.0 L119.2,427.8 L119.8,427.7 L120.3,427.5 L120.9,427.4 L121.5,427.2 L122.1,427.1 L122.7,427.0
 L123.3,426.8 L123.8,426.7 L124.4,426.6 L125.0,426.4 L125.6,426.3 L126.2,426.2 L126.8,426.1 L127.3,426.0
 L127.9,425.8 L128.5,425.7 L129.1,425.6 L129.7,425.5 L130.3,425.5 L130.8,425.4 L131.4,425.3 L132.0,425.2
 L132.6,425.1 L133.2,425.1 L133.8,425.0 L134.3,424.9 L134.9,424.9 L135.5,424.8 L136.1,424.8 L136.7,424.7
 L137.3,424.7 L137.8,424.7 L138.4,424.6 L139.0,424.6 L139.6,424.6 L140.2,424.6 L140.8,424.6 L141.3,424.6
 L141.9,424.6 L142.5,424.6 L143.1,424.6 L143.7,424.6 L144.2,424.6 L144.8,424.6 L145.4,424.7 L146.0,424.7
 L146.6,424.7 L147.2,424.8 L147.7,424.8 L148.3,424.9 L148.9,424.9 L149.5,425.0 L150.1,425.1 L150.7,425.1
 L151.2,425.2 L151.8,425.3 L152.4,425.4 L153.0,425.5 L153.6,425.6 L154.2,425.6 L154.7,425.7 L155.3,425.8
 L155.9,426.0 L156.5,426.1 L157.1,426.2 L157.7,426.3 L158.2,426.4 L158.8,426.5 L159.4,426.7 L160.0,426.8
 L160.6,426.9 L161.2,427.1 L161.7,427.2 L162.3,427.3 L162.9,427.5 L163.5,427.6 L164.1,427.8 L164.6,427.9
 L165.2,428.1 L165.8,428.2 L166.4,428.4 L167.0,428.5 L167.6,428.7 L168.1,428.8 L168.7,429.0 L169.3,429.2
 L169.9,429.3 L170.5,429.5 L171.1,429.6 L171.6,429.8 L172.2,430.0 L172.8,430.1 L173.4,430.3 L174.0,430.5
 L174.6,430.6 L175.1,430.8 L175.7,431.0 L176.3,431.1 L176.9,431.3 L177.5,431.5 L178.1,431.6 L178.6,431.8
 L179.2,432.0 L179.8,432.1 L180.4,432.3 L181.0,432.5 L181.6,432.6 L182.1,432.8 L182.7,432.9 L183.3,433.1
 L183.9,433.2 L184.5,433.4 L185.0,433.5 L185.6,433.7 L186.2,433.8 L186.8,434.0 L187.4,434.1 L188.0,434.3
 L188.5,434.4 L189.1,434.5 L189.7,434.7 L190.3,434.8 L190.9,434.9 L191.5,435.0 L192.0,435.2 L192.6,435.3
 L193.2,435.4 L193.8,435.5 L194.4,435.6 L195.0,435.7 L195.5,435.8 L196.1,435.9 L196.7,436.0 L197.3,436.1
 L197.9,436.2 L198.5,436.3 L199.0,436.4 L199.6,436.5 L200.2,436.5 L200.8,436.6 L201.4,436.7 L202.0,436.7
 L202.5,436.8 L203.1,436.9 L203.7,436.9 L204.3,437.0 L204.9,437.0 L205.4,437.1 L206.0,437.1 L206.6,437.2
 L207.2,437.2 L207.8,437.2 L208.4,437.3 L208.9,437.3 L209.5,437.3 L210.1,437.3 L210.7,437.3 L211.3,437.3
 L211.9,437.3 L212.4,437.4 L213.0,437.4 L213.6,437.4 L214.2,437.4 L214.8,437.3 L215.4,437.3 L215.9,437.3
 L216.5,437.3 L217.1,437.3 L217.7,437.3 L218.3,437.2 L218.9,437.2 L219.4,437.2 L220.0,437.1 L220.6,437.1
 L221.2,437.1 L221.8,437.0 L222.4,437.0 L222.

In [68]:
public static void CheckAgainstOriginEnergyData(List<ISessionInfo> StudySess, List<ISessionInfo> OriginSess, string mStudy, string EnergyName, double errThrsh, bool output = false) {

    foreach (var study in StudySess) {

        if (!study.Name.Contains(mStudy)) {
            continue;
        }

        List<ISessionInfo> readSess = new List<ISessionInfo>() { study };
        Plot2Ddata readData;
        try {
            readData = readSess.ReadLogDataValue( "Energy", EnergyName, output: false);
        } catch {
            readData = readSess.ReadLogDataValue( "EnergyLogValues", EnergyName, output: false);
        }       
        var Sgrp = readData.dataGroups[0];

        readSess = OriginSess.Where(grp => grp.Name.Contains(study.Name)).ToList<ISessionInfo>();
        try {
            readData = readSess.ReadLogDataValue( "Energy", EnergyName, output: false);
        } catch {
            readData = readSess.ReadLogDataValue( "EnergyLogValues", EnergyName, output: false);
        }       
        var Ogrp = readData.dataGroups[0];

        if (Sgrp.Abscissas.Length == 0 || Ogrp.Abscissas.Length == 0) {
            Console.WriteLine("no data available - skipping session ...");
            continue;
        }

        Console.WriteLine($"checking session {study.Name}"); 

        var errorData = ISessionInfoExtensions.ComputeErrorNormsForLogData(Sgrp, Ogrp);

        if (output) {
            Console.WriteLine($"    l1-error norm = {errorData["l_1 error norm"]}");
            Console.WriteLine($"    l2-error norm = {errorData["l_2 error norm"]}");
        }
        
        NUnit.Framework.Assert.LessOrEqual(errorData["l_1 error norm"], errThrsh,  
            $"l_1 error norm larger than allowed: {errorData["l_1 error norm"]}");

        NUnit.Framework.Assert.LessOrEqual(errorData["l_2 error norm"], errThrsh,  
            $"l_2 error norm larger than allowed: {errorData["l_2 error norm"]}");
    }

}

In [74]:
CheckAgainstOriginEnergyData(evalSess, evalSessOrigin, "m2", "kineticEnergy", 0.0003, false)

checking session OD3D_m2_Oh01_eta01_stagnantInit
checking session OD3D_m2_Oh01_eta02_stagnantInit
checking session OD3D_m2_Oh01_eta03_stagnantInit
checking session OD3D_m2_Oh01_eta04_stagnantInit
checking session OD3D_m2_Oh01_eta05_stagnantInit
checking session OD3D_m2_Oh01_eta06_stagnantInit
checking session OD3D_m2_Oh01_eta07_stagnantInit


In [73]:
CheckAgainstOriginEnergyData(evalSess, evalSessOrigin, "m3", "kineticEnergy", 0.002, false)

checking session OD3D_m3_Oh01_eta015_stagnantInit
checking session OD3D_m3_Oh01_eta03_stagnantInit
checking session OD3D_m3_Oh01_eta04_stagnantInit
no data available - skipping session ...


In [72]:
CheckAgainstOriginEnergyData(evalSess, evalSessOrigin, "m4", "kineticEnergy", 0.003, false)

checking session OD3D_m4_Oh01_eta01_stagnantInit
checking session OD3D_m4_Oh01_eta02_stagnantInit
checking session OD3D_m4_Oh01_eta03_stagnantInit
checking session OD3D_m4_Oh01_eta04_stagnantInit
checking session OD3D_m4_Oh01_eta05_stagnantInit
checking session OD3D_m4_Oh01_eta06_stagnantInit
checking session OD3D_m4_Oh01_eta07_stagnantInit


### Fig. 22 
Surface energy deviation from the spherical state $\Delta E_{surf} = \int_{\mathfrak{I}} 1 dA - 4\pi $ over time $t$ for the different modes of initial deformation
(a) $m = 2$, (b) $m = 3$, and (c) $m = 4$, with deformation parameters up to $\eta_0 = 0.7$ and Ohnesorge number
$Oh = 0.1$. The red dash-dotted line indicates WNLT results.


In [52]:
Plot2Ddata[,] Fig22 = new Plot2Ddata[2, 2];

//var fig = GetAspectRatioOverTime_Plot2Ddata(evalStudyData, new string[] { "m2-Oh01-eta01-stagnantInit", "m2-Oh01-eta01-thirdOrderInit" });
var fig = GetEnergy_Plot2Ddata(plotEvalSess, "surfaceEnergy",
            new string[] { "m2_Oh01_eta01_stagnantInit", "m2_Oh01_eta02_stagnantInit", "m2_Oh01_eta03_stagnantInit", "m2_Oh01_eta04_stagnantInit",
                            "m2_Oh01_eta05_stagnantInit", "m2_Oh01_eta06_stagnantInit", "m2_Oh01_eta07_stagnantInit" },
            E0: 4.0 * Math.PI,
            legendNames: new string[] { "eta_0 = 0.1", "eta_0 = 0.2", "eta_0 = 0.3", "eta_0 = 0.4", "eta_0 = 0.5", "eta_0 = 0.6", "eta_0 = 0.7" });
fig.YrangeMin = 0.0;
fig.YrangeMax = 2.0;
Fig22[0, 0] = fig; 

//var fig = GetAspectRatioOverTime_Plot2Ddata(evalStudyData, new string[] { "m2-Oh01-eta01-stagnantInit", "m2-Oh01-eta01-thirdOrderInit" });
fig = GetEnergy_Plot2Ddata(plotEvalSess, "surfaceEnergy",
            new string[] { "m3_Oh01_eta015_stagnantInit", "m3_Oh01_eta03_stagnantInit", "m3_Oh01_eta04_stagnantInit", "m3_Oh01_eta05_stagnantInit" },
            E0: 4.0 * Math.PI,
            legendNames: new string[] { "eta_0 = 0.15", "eta_0 = 0.3", "eta_0 = 0.4", "eta_0 = 0.5" });
fig.YrangeMin = 0.0;
fig.YrangeMax = 2.0;
Fig22[0, 1] = fig;

//var fig = GetAspectRatioOverTime_Plot2Ddata(evalStudyData, new string[] { "m2-Oh01-eta01-stagnantInit", "m2-Oh01-eta01-thirdOrderInit" });
fig = GetEnergy_Plot2Ddata(plotEvalSess, "surfaceEnergy",
            new string[] { "m4_Oh01_eta01_stagnantInit", "m4_Oh01_eta02_stagnantInit", "m4_Oh01_eta03_stagnantInit", "m4_Oh01_eta04_stagnantInit",
                            "m4_Oh01_eta05_stagnantInit", "m4_Oh01_eta06_stagnantInit", "m4_Oh01_eta07_stagnantInit" },
            E0: 4.0 * Math.PI,
            legendNames: new string[] { "eta_0 = 0.1", "eta_0 = 0.2", "eta_0 = 0.3", "eta_0 = 0.4", "eta_0 = 0.5", "eta_0 = 0.6", "eta_0 = 0.7" });
fig.YrangeMin = 0.0;
fig.YrangeMax = 5.0;
Fig22[1, 0] = fig;

Fig22.ToGnuplot().PlotSVG(xRes:1200,yRes:1000)

Processing session: OD3D_m2_Oh01_eta01_stagnantInit_restart3
  Found restart session: a2d5a18a-39f4-449a-8013-54106406c1cf
  Found restart session: 2a51c1d4-cf2c-45f6-9baf-be68918ddc54
  Found restart session: 59bb73dc-0683-42a1-a615-e87ed3524000
  Merging data from 4 sessions.
Processing session: OD3D_m2_Oh01_eta01_stagnantInit_restart3
  Found restart session: a2d5a18a-39f4-449a-8013-54106406c1cf
  Found restart session: 2a51c1d4-cf2c-45f6-9baf-be68918ddc54
  Found restart session: 59bb73dc-0683-42a1-a615-e87ed3524000
  Merging data from 4 sessions.
... Done
Build DataSet
Processing session: OD3D_m2_Oh01_eta02_stagnantInit_restart3
  Found restart session: 33d67916-8020-4464-a16e-f55562a81bb0
  Found restart session: 6b199ada-b490-49b1-99d1-0a7ec09620fd
  Found restart session: 2e275baf-7a62-4b31-a3af-e15cf125f40c
  Merging data from 4 sessions.
Processing session: OD3D_m2_Oh01_eta02_stagnantInit_restart3
  Found restart session: 33d67916-8020-4464-a16e-f55562a81bb0
  Found restart s

<?xml version="1.0" encoding="utf-8" standalone="no"?>
<!DOCTYPE svg PUBLIC "-//W3C//DTD SVG 1.1//EN"
 "http://www.w3.org/Graphics/SVG/1.1/DTD/svg11.dtd">
 

 Gnuplot 
 Produced by GNUPLOT 5.1 patchlevel 0 

 

 
 

 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 0 
 
 
 
 
 0.5 
 
 
 
 
 1 
 
 
 
 
 1.5 
 
 
 
 
 2 
 
 
 
 
 0 
 
 
 
 
 0.5 
 
 
 
 
 1 
 
 
 
 
 1.5 
 
 
 
 
 2 
 
 
 
 
 2.5 
 
 
 
 
 3 
 
 
 
 
 3.5 
 
 
 
 
 4 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 surfaceEnergy 
 
 
 
 
 Time 
 
 
 
 
 eta0 = 0.1 
 
 
 
 
 eta 0 = 0.1 
 
 
 
	<path stroke='rgb(255, 0, 0)' d='M371.4,62.1 L424.8,62.1 M77.8,427.7 L78.4,427.7 L79.0,427.7 L79.5,427.7 L80.1,427.7 L80.7,427.7
 L81.3,427.7 L81.9,427.7 L82.5,427.8 L83.0,427.8 L83.6,427.8 L84.2,427.9 L84.8,427.9 L85.4,427.9
 L86.0,428.0 L86.5,428.0 L87.1,428.1 L87.7,428.1 L88.3,428.2 L88.9,428.2 L89.5,428.3 L90.0,428.4
 L90.6,428.4 L91.2,428.5 L91.8,428.6 L92.4,428.6 L93.0,428.7 L93.5,428.8 L94.1,428.9 L94.7,428.9
 L95.3,429.0 L95.9,429.1 L96.5,429.2 L97.0,429.3 L97.6,429.4 L98.2,429.5 L98.8,429.6 L99.4,429.7
 L99.9,429.8 L100.5,429.9 L101.1,430.0 L101.7,430.1 L102.3,430.2 L102.9,430.3 L103.4,430.4 L104.0,430.5
 L104.6,430.6 L105.2,430.7 L105.8,430.8 L106.4,431.0 L106.9,431.1 L107.5,431.2 L108.1,431.3 L108.7,431.4
 L109.3,431.5 L109.9,431.6 L110.4,431.8 L111.0,431.9 L111.6,432.0 L112.2,432.1 L112.8,432.2 L113.4,432.3
 L113.9,432.5 L114.5,432.6 L115.1,432.7 L115.7,432.8 L116.3,432.9 L116.9,433.1 L117.4,433.2 L118.0,433.3
 L118.6,433.4 L119.2,433.5 L119.8,433.6 L120.3,433.7 L120.9,433.9 L121.5,434.0 L122.1,434.1 L122.7,434.2
 L123.3,434.3 L123.8,434.4 L124.4,434.5 L125.0,434.6 L125.6,434.7 L126.2,434.8 L126.8,434.9 L127.3,435.0
 L127.9,435.1 L128.5,435.2 L129.1,435.3 L129.7,435.4 L130.3,435.5 L130.8,435.6 L131.4,435.7 L132.0,435.8
 L132.6,435.9 L133.2,435.9 L133.8,436.0 L134.3,436.1 L134.9,436.2 L135.5,436.3 L136.1,436.3 L136.7,436.4
 L137.3,436.5 L137.8,436.5 L138.4,436.6 L139.0,436.7 L139.6,436.7 L140.2,436.8 L140.8,436.8 L141.3,436.9
 L141.9,436.9 L142.5,437.0 L143.1,437.0 L143.7,437.1 L144.2,437.1 L144.8,437.2 L145.4,437.2 L146.0,437.2
 L146.6,437.2 L147.2,437.3 L147.7,437.3 L148.3,437.3 L148.9,437.3 L149.5,437.4 L150.1,437.4 L150.7,437.4
 L151.2,437.4 L151.8,437.4 L152.4,437.4 L153.0,437.4 L153.6,437.4 L154.2,437.4 L154.7,437.4 L155.3,437.4
 L155.9,437.4 L156.5,437.4 L157.1,437.4 L157.7,437.4 L158.2,437.4 L158.8,437.3 L159.4,437.3 L160.0,437.3
 L160.6,437.3 L161.2,437.2 L161.7,437.2 L162.3,437.2 L162.9,437.1 L163.5,437.1 L164.1,437.1 L164.6,437.0
 L165.2,437.0 L165.8,437.0 L166.4,436.9 L167.0,436.9 L167.6,436.8 L168.1,436.8 L168.7,436.7 L169.3,436.7
 L169.9,436.6 L170.5,436.6 L171.1,436.5 L171.6,436.5 L172.2,436.4 L172.8,436.4 L173.4,436.3 L174.0,436.3
 L174.6,436.2 L175.1,436.1 L175.7,436.1 L176.3,436.0 L176.9,436.0 L177.5,435.9 L178.1,435.8 L178.6,435.8
 L179.2,435.7 L179.8,435.7 L180.4,435.6 L181.0,435.5 L181.6,435.5 L182.1,435.4 L182.7,435.4 L183.3,435.3
 L183.9,435.2 L184.5,435.2 L185.0,435.1 L185.6,435.1 L186.2,435.0 L186.8,434.9 L187.4,434.9 L188.0,434.8
 L188.5,434.8 L189.1,434.7 L189.7,434.7 L190.3,434.6 L190.9,434.6 L191.5,434.5 L192.0,434.4 L192.6,434.4
 L193.2,434.3 L193.8,434.3 L194.4,434.3 L195.0,434.2 L195.5,434.2 L196.1,434.1 L196.7,434.1 L197.3,434.0
 L197.9,434.0 L198.5,434.0 L199.0,433.9 L199.6,433.9 L200.2,433.8 L200.8,433.8 L201.4,433.8 L202.0,433.7
 L202.5,433.7 L203.1,433.7 L203.7,433.7 L204.3,433.6 L204.9,433.6 L205.4,433.6 L206.0,433.6 L206.6,433.6
 L207.2,433.5 L207.8,433.5 L208.4,433.5 L208.9,433.5 L209.5,433.5 L210.1,433.5 L210.7,433.5 L211.3,433.5
 L211.9,433.5 L212.4,433.5 L213.0,433.5 L213.6,433.5 L214.2,433.5 L214.8,433.5 L215.4,433.5 L215.9,433.5
 L216.5,433.5 L217.1,433.5 L217.7,433.5 L218.3,433.6 L218.9,433.6 L219.4,433.6 L220.0,433.6 L220.6,433.6
 L221.2,433.7 L221.8,433.7 L222.4,433.7 L222.9,433.7 L223.5,433.8 L224.

In [79]:
CheckAgainstOriginEnergyData(evalSess, evalSessOrigin, "m2", "surfaceEnergy", 0.0001, false)

checking session OD3D_m2_Oh01_eta01_stagnantInit
checking session OD3D_m2_Oh01_eta02_stagnantInit
checking session OD3D_m2_Oh01_eta03_stagnantInit
checking session OD3D_m2_Oh01_eta04_stagnantInit
checking session OD3D_m2_Oh01_eta05_stagnantInit
checking session OD3D_m2_Oh01_eta06_stagnantInit
checking session OD3D_m2_Oh01_eta07_stagnantInit


In [82]:
CheckAgainstOriginEnergyData(evalSess, evalSessOrigin, "m3", "surfaceEnergy", 0.00003, false)

checking session OD3D_m3_Oh01_eta015_stagnantInit
checking session OD3D_m3_Oh01_eta03_stagnantInit
checking session OD3D_m3_Oh01_eta04_stagnantInit
no data available - skipping session ...


In [89]:
CheckAgainstOriginEnergyData(evalSess, evalSessOrigin, "m4", "surfaceEnergy", 0.0006, false)

checking session OD3D_m4_Oh01_eta01_stagnantInit
checking session OD3D_m4_Oh01_eta02_stagnantInit
checking session OD3D_m4_Oh01_eta03_stagnantInit
checking session OD3D_m4_Oh01_eta04_stagnantInit
checking session OD3D_m4_Oh01_eta05_stagnantInit
checking session OD3D_m4_Oh01_eta06_stagnantInit
checking session OD3D_m4_Oh01_eta07_stagnantInit


### Fig. 23 
Total energy deviation from the spherical state $\Delta E_{tot} = E_{kin} + \Delta E_{surf}$ over time $t$ for the different modes of initial deformation
(a) $m = 2$, (b) $m = 3$, and (c) $m = 4$, with deformation parameters up to $\eta_0 = 0.7$ and Ohnesorge number
$Oh = 0.1$. The red dash-dotted line indicates WNLT results.


In [53]:
Plot2Ddata[,] Fig23 = new Plot2Ddata[2, 2];

//var fig = GetAspectRatioOverTime_Plot2Ddata(evalStudyData, new string[] { "m2-Oh01-eta01-stagnantInit", "m2-Oh01-eta01-thirdOrderInit" });
var fig = GetEnergy_Plot2Ddata(plotEvalSess, "totalEnergy",
            new string[] { "m2_Oh01_eta01_stagnantInit", "m2_Oh01_eta02_stagnantInit", "m2_Oh01_eta03_stagnantInit", "m2_Oh01_eta04_stagnantInit",
                            "m2_Oh01_eta05_stagnantInit", "m2_Oh01_eta06_stagnantInit", "m2_Oh01_eta07_stagnantInit" },
            E0: 4.0 * Math.PI,
            legendNames: new string[] { "eta_0 = 0.1", "eta_0 = 0.2", "eta_0 = 0.3", "eta_0 = 0.4", "eta_0 = 0.5", "eta_0 = 0.6", "eta_0 = 0.7" });
fig.YrangeMin = 0.0;
fig.YrangeMax = 2.0;
Fig23[0, 0] = fig; 

//var fig = GetAspectRatioOverTime_Plot2Ddata(evalStudyData, new string[] { "m2-Oh01-eta01-stagnantInit", "m2-Oh01-eta01-thirdOrderInit" });
fig = GetEnergy_Plot2Ddata(plotEvalSess, "totalEnergy",
            new string[] { "m3_Oh01_eta015_stagnantInit", "m3_Oh01_eta03_stagnantInit", "m3_Oh01_eta04_stagnantInit", "m3_Oh01_eta05_stagnantInit" },
            E0: 4.0 * Math.PI,
            legendNames: new string[] { "eta_0 = 0.15", "eta_0 = 0.3", "eta_0 = 0.4", "eta_0 = 0.5" });
fig.YrangeMin = 0.0;
fig.YrangeMax = 2.0;
Fig23[0, 1] = fig;

//var fig = GetAspectRatioOverTime_Plot2Ddata(evalStudyData, new string[] { "m2-Oh01-eta01-stagnantInit", "m2-Oh01-eta01-thirdOrderInit" });
fig = GetEnergy_Plot2Ddata(plotEvalSess, "totalEnergy",
            new string[] { "m4_Oh01_eta01_stagnantInit", "m4_Oh01_eta02_stagnantInit", "m4_Oh01_eta03_stagnantInit", "m4_Oh01_eta04_stagnantInit",
                            "m4_Oh01_eta05_stagnantInit", "m4_Oh01_eta06_stagnantInit", "m4_Oh01_eta07_stagnantInit" },
            E0: 4.0 * Math.PI,
            legendNames: new string[] { "eta_0 = 0.1", "eta_0 = 0.2", "eta_0 = 0.3", "eta_0 = 0.4", "eta_0 = 0.5", "eta_0 = 0.6", "eta_0 = 0.7" });
fig.YrangeMin = 0.0;
fig.YrangeMax = 5.0;
Fig23[1, 0] = fig;

Fig23.ToGnuplot().PlotSVG(xRes:1200,yRes:1000)

Processing session: OD3D_m2_Oh01_eta01_stagnantInit_restart3
  Found restart session: a2d5a18a-39f4-449a-8013-54106406c1cf
  Found restart session: 2a51c1d4-cf2c-45f6-9baf-be68918ddc54
  Found restart session: 59bb73dc-0683-42a1-a615-e87ed3524000
  Merging data from 4 sessions.
Processing session: OD3D_m2_Oh01_eta01_stagnantInit_restart3
  Found restart session: a2d5a18a-39f4-449a-8013-54106406c1cf
  Found restart session: 2a51c1d4-cf2c-45f6-9baf-be68918ddc54
  Found restart session: 59bb73dc-0683-42a1-a615-e87ed3524000
  Merging data from 4 sessions.
... Done
Build DataSet
Processing session: OD3D_m2_Oh01_eta02_stagnantInit_restart3
  Found restart session: 33d67916-8020-4464-a16e-f55562a81bb0
  Found restart session: 6b199ada-b490-49b1-99d1-0a7ec09620fd
  Found restart session: 2e275baf-7a62-4b31-a3af-e15cf125f40c
  Merging data from 4 sessions.
Processing session: OD3D_m2_Oh01_eta02_stagnantInit_restart3
  Found restart session: 33d67916-8020-4464-a16e-f55562a81bb0
  Found restart s

<?xml version="1.0" encoding="utf-8" standalone="no"?>
<!DOCTYPE svg PUBLIC "-//W3C//DTD SVG 1.1//EN"
 "http://www.w3.org/Graphics/SVG/1.1/DTD/svg11.dtd">
 

 Gnuplot 
 Produced by GNUPLOT 5.1 patchlevel 0 

 

 
 

 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 0 
 
 
 
 
 0.5 
 
 
 
 
 1 
 
 
 
 
 1.5 
 
 
 
 
 2 
 
 
 
 
 0 
 
 
 
 
 0.5 
 
 
 
 
 1 
 
 
 
 
 1.5 
 
 
 
 
 2 
 
 
 
 
 2.5 
 
 
 
 
 3 
 
 
 
 
 3.5 
 
 
 
 
 4 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 totalEnergy 
 
 
 
 
 Time 
 
 
 
 
 eta0 = 0.1 
 
 
 
 
 eta 0 = 0.1 
 
 
 
	<path stroke='rgb(255, 0, 0)' d='M371.4,62.1 L424.8,62.1 M77.8,427.7 L78.4,427.7 L79.0,427.7 L79.5,427.7 L80.1,427.7 L80.7,427.7
 L81.3,427.7 L81.9,427.7 L82.5,427.7 L83.0,427.7 L83.6,427.7 L84.2,427.7 L84.8,427.7 L85.4,427.7
 L86.0,427.7 L86.5,427.7 L87.1,427.7 L87.7,427.7 L88.3,427.7 L88.9,427.7 L89.5,427.7 L90.0,427.7
 L90.6,427.7 L91.2,427.7 L91.8,427.7 L92.4,427.7 L93.0,427.7 L93.5,427.7 L94.1,427.7 L94.7,427.7
 L95.3,427.7 L95.9,427.8 L96.5,427.8 L97.0,427.8 L97.6,427.8 L98.2,427.8 L98.8,427.8 L99.4,427.8
 L99.9,427.8 L100.5,427.9 L101.1,427.9 L101.7,427.9 L102.3,427.9 L102.9,427.9 L103.4,427.9 L104.0,428.0
 L104.6,428.0 L105.2,428.0 L105.8,428.0 L106.4,428.0 L106.9,428.1 L107.5,428.1 L108.1,428.1 L108.7,428.1
 L109.3,428.2 L109.9,428.2 L110.4,428.2 L111.0,428.2 L111.6,428.3 L112.2,428.3 L112.8,428.3 L113.4,428.4
 L113.9,428.4 L114.5,428.4 L115.1,428.4 L115.7,428.5 L116.3,428.5 L116.9,428.5 L117.4,428.6 L118.0,428.6
 L118.6,428.7 L119.2,428.7 L119.8,428.7 L120.3,428.8 L120.9,428.8 L121.5,428.8 L122.1,428.9 L122.7,428.9
 L123.3,429.0 L123.8,429.0 L124.4,429.0 L125.0,429.1 L125.6,429.1 L126.2,429.2 L126.8,429.2 L127.3,429.3
 L127.9,429.3 L128.5,429.4 L129.1,429.4 L129.7,429.4 L130.3,429.5 L130.8,429.5 L131.4,429.6 L132.0,429.6
 L132.6,429.7 L133.2,429.7 L133.8,429.8 L134.3,429.8 L134.9,429.9 L135.5,429.9 L136.1,430.0 L136.7,430.0
 L137.3,430.1 L137.8,430.1 L138.4,430.2 L139.0,430.2 L139.6,430.3 L140.2,430.3 L140.8,430.4 L141.3,430.4
 L141.9,430.5 L142.5,430.5 L143.1,430.6 L143.7,430.6 L144.2,430.7 L144.8,430.7 L145.4,430.8 L146.0,430.8
 L146.6,430.9 L147.2,430.9 L147.7,431.0 L148.3,431.0 L148.9,431.1 L149.5,431.1 L150.1,431.2 L150.7,431.2
 L151.2,431.3 L151.8,431.3 L152.4,431.4 L153.0,431.4 L153.6,431.4 L154.2,431.5 L154.7,431.5 L155.3,431.6
 L155.9,431.6 L156.5,431.7 L157.1,431.7 L157.7,431.8 L158.2,431.8 L158.8,431.8 L159.4,431.9 L160.0,431.9
 L160.6,432.0 L161.2,432.0 L161.7,432.1 L162.3,432.1 L162.9,432.1 L163.5,432.2 L164.1,432.2 L164.6,432.2
 L165.2,432.3 L165.8,432.3 L166.4,432.4 L167.0,432.4 L167.6,432.4 L168.1,432.5 L168.7,432.5 L169.3,432.5
 L169.9,432.5 L170.5,432.6 L171.1,432.6 L171.6,432.6 L172.2,432.7 L172.8,432.7 L173.4,432.7 L174.0,432.7
 L174.6,432.8 L175.1,432.8 L175.7,432.8 L176.3,432.9 L176.9,432.9 L177.5,432.9 L178.1,432.9 L178.6,432.9
 L179.2,433.0 L179.8,433.0 L180.4,433.0 L181.0,433.0 L181.6,433.0 L182.1,433.1 L182.7,433.1 L183.3,433.1
 L183.9,433.1 L184.5,433.1 L185.0,433.1 L185.6,433.2 L186.2,433.2 L186.8,433.2 L187.4,433.2 L188.0,433.2
 L188.5,433.2 L189.1,433.2 L189.7,433.2 L190.3,433.2 L190.9,433.3 L191.5,433.3 L192.0,433.3 L192.6,433.3
 L193.2,433.3 L193.8,433.3 L194.4,433.3 L195.0,433.3 L195.5,433.3 L196.1,433.3 L196.7,433.3 L197.3,433.3
 L197.9,433.3 L198.5,433.3 L199.0,433.4 L199.6,433.4 L200.2,433.4 L200.8,433.4 L201.4,433.4 L202.0,433.4
 L202.5,433.4 L203.1,433.4 L203.7,433.4 L204.3,433.4 L204.9,433.4 L205.4,433.4 L206.0,433.4 L206.6,433.4
 L207.2,433.4 L207.8,433.4 L208.4,433.4 L208.9,433.4 L209.5,433.4 L210.1,433.4 L210.7,433.4 L211.3,433.4
 L211.9,433.4 L212.4,433.4 L213.0,433.4 L213.6,433.4 L214.2,433.4 L214.8,433.4 L215.4,433.4 L215.9,433.4
 L216.5,433.4 L217.1,433.4 L217.7,433.4 L218.3,433.4 L218.9,433.4 L219.4,433.4 L220.0,433.4 L220.6,433.4
 L221.2,433.4 L221.8,433.4 L222.4,433.4 L222.9,433.5 L223.5,433.5 L224.1,

In [90]:
CheckAgainstOriginEnergyData(evalSess, evalSessOrigin, "m2", "totalEnergy", 0.0003, false)

checking session OD3D_m2_Oh01_eta01_stagnantInit
checking session OD3D_m2_Oh01_eta02_stagnantInit
checking session OD3D_m2_Oh01_eta03_stagnantInit
checking session OD3D_m2_Oh01_eta04_stagnantInit
checking session OD3D_m2_Oh01_eta05_stagnantInit
checking session OD3D_m2_Oh01_eta06_stagnantInit
checking session OD3D_m2_Oh01_eta07_stagnantInit


In [91]:
CheckAgainstOriginEnergyData(evalSess, evalSessOrigin, "m3", "totalEnergy", 0.002, false)

checking session OD3D_m3_Oh01_eta015_stagnantInit
checking session OD3D_m3_Oh01_eta03_stagnantInit
checking session OD3D_m3_Oh01_eta04_stagnantInit
no data available - skipping session ...


In [92]:
CheckAgainstOriginEnergyData(evalSess, evalSessOrigin, "m4", "totalEnergy", 0.003, false)

checking session OD3D_m4_Oh01_eta01_stagnantInit
checking session OD3D_m4_Oh01_eta02_stagnantInit
checking session OD3D_m4_Oh01_eta03_stagnantInit
checking session OD3D_m4_Oh01_eta04_stagnantInit
checking session OD3D_m4_Oh01_eta05_stagnantInit
checking session OD3D_m4_Oh01_eta06_stagnantInit
checking session OD3D_m4_Oh01_eta07_stagnantInit


### North Poles

In [54]:
public static double[] GetNorthPoleByModeDecomposition(Plot2Ddata modeData) {

    int numTimesteps = modeData.dataGroups[0].Abscissas.Length;
    double[] northPolePositions = new double[numTimesteps];

    for (int ts = 0; ts < numTimesteps; ts++) {

        double northPole = 0.0;

        int maxL = modeData.dataGroups.Count();
        for (int m = 0; m < maxL; m++) {

            if (m == 1)
                continue;

            double mValue = modeData.dataGroups[m].Values[ts];

            northPole += mValue * SphericalHarmonics.MyLegendre(m,0,Math.Cos(0.0));        
        }
        northPolePositions[ts] = northPole;
    }

    return northPolePositions;
}

In [55]:
public static Plot2Ddata GetNorthPoleStudyOverTimeByModeDecomposition_Plot2Ddata(List<Plot2Ddata> modeDataS, string[] studies, string[] legendNames = null) {

    Plot2Ddata plt =  new Plot2Ddata();
    plt.Xlabel = "t";
    plt.Ylabel = "L/W";

    int lcIdx = 0;
    for (int i = 0; i < studies.Length; i++) {
        var datPlt = modeDataS.Where(plt => plt.Title.Contains(studies[i])).Single();

        double[] northPole = GetNorthPoleByModeDecomposition(datPlt);
    
        var fmt = new PlotFormat();
        fmt.Style = Styles.Lines;
        fmt.DashType = DashTypes.Solid;
        fmt.LineWidth = 2;
        //LineColors[] customColors = new LineColors[] { LineColors.Blue, LineColors.Red, LineColors.Green };
        fmt.LineColor = (LineColors)(lcIdx);

        string[] nameParts = datPlt.Title.Split(new string[] { "-" }, StringSplitOptions.RemoveEmptyEntries);
        string name = legendNames != null ? legendNames[i] : nameParts[3];
        plt.AddDataGroup($"{name}", datPlt.dataGroups[0].Abscissas, northPole, fmt);

        lcIdx++;      
    }   

    plt.XrangeMin = 0.0;
    plt.XrangeMax = 4.0;

    plt.ShowLegend = true;

    return plt;        
}

### FIG. 25.
Droplet north pole position $r_s(0,t)$ over time $t$ for drops with $Oh = 0.1$ and different degrees of
initial deformation in the modes of initial deformation (a) $m = 2$, (b) $m = 3$, and (c) $m = 4$. The deformation
parameters up to $\eta_0 = 0.7$ are covered.

In [56]:
Plot2Ddata[,] Fig25 = new Plot2Ddata[2, 2];

//var fig = GetAspectRatioOverTime_Plot2Ddata(evalStudyData, new string[] { "m2-Oh01-eta01-stagnantInit", "m2-Oh01-eta01-thirdOrderInit" });
var fig = GetNorthPoleStudyOverTimeByModeDecomposition_Plot2Ddata(plotModeData, 
            new string[] { "m2-Oh01-eta01-stagnantInit", "m2-Oh01-eta02-stagnantInit", "m2-Oh01-eta03-stagnantInit", "m2-Oh01-eta04-stagnantInit",
                            "m2-Oh01-eta05-stagnantInit", "m2-Oh01-eta06-stagnantInit", "m2-Oh01-eta07-stagnantInit" },
            new string[] { "eta_0 = 0.1", "eta_0 = 0.2", "eta_0 = 0.3", "eta_0 = 0.4", "eta_0 = 0.5", "eta_0 = 0.6", "eta_0 = 0.7" });
fig.YrangeMin = 0.6;
fig.YrangeMax = 2.0;
Fig25[0, 0] = fig; 

//var fig = GetAspectRatioOverTime_Plot2Ddata(evalStudyData, new string[] { "m2-Oh01-eta01-stagnantInit", "m2-Oh01-eta01-thirdOrderInit" });
fig = GetNorthPoleStudyOverTimeByModeDecomposition_Plot2Ddata(plotModeData, 
            new string[] { "m3-Oh01-eta015-stagnantInit", "m3-Oh01-eta03-stagnantInit", "m3-Oh01-eta04-stagnantInit", "m3-Oh01-eta05-stagnantInit" },
            new string[] { "eta_0 = 0.15", "eta_0 = 0.3", "eta_0 = 0.4", "eta_0 = 0.5" });
fig.YrangeMin = 0.8;
fig.YrangeMax = 1.5;
Fig25[0, 1] = fig;

//var fig = GetAspectRatioOverTime_Plot2Ddata(evalStudyData, new string[] { "m2-Oh01-eta01-stagnantInit", "m2-Oh01-eta01-thirdOrderInit" });
fig = GetNorthPoleStudyOverTimeByModeDecomposition_Plot2Ddata(plotModeData, 
            new string[] { "m4-Oh01-eta01-stagnantInit", "m4-Oh01-eta02-stagnantInit", "m4-Oh01-eta03-stagnantInit", "m4-Oh01-eta04-stagnantInit",
                            "m4-Oh01-eta05-stagnantInit", "m4-Oh01-eta06-stagnantInit", "m4-Oh01-eta07-stagnantInit" },
            new string[] { "eta_0 = 0.1", "eta_0 = 0.2", "eta_0 = 0.3", "eta_0 = 0.4", "eta_0 = 0.5", "eta_0 = 0.6", "eta_0 = 0.7" });
fig.YrangeMin = 0.8;
fig.YrangeMax = 1.8;
Fig25[1, 0] = fig;

Fig25.ToGnuplot().PlotSVG(xRes:1200,yRes:1000)

Using gnuplot: C:\Program Files (x86)\FDY\BoSSS\bin\native\win\gnuplot-gp510-20160418-win32-mingw\gnuplot\bin\gnuplot.exe
Note: In a Jupyter Worksheet, you must NOT have a trailing semicolon in order to see the plot on screen; otherwise, the output migth be surpressed.!


<?xml version="1.0" encoding="utf-8" standalone="no"?>
<!DOCTYPE svg PUBLIC "-//W3C//DTD SVG 1.1//EN"
 "http://www.w3.org/Graphics/SVG/1.1/DTD/svg11.dtd">
 

 Gnuplot 
 Produced by GNUPLOT 5.1 patchlevel 0 

 

 
 

 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 0.6 
 
 
 
 
 0.8 
 
 
 
 
 1 
 
 
 
 
 1.2 
 
 
 
 
 1.4 
 
 
 
 
 1.6 
 
 
 
 
 1.8 
 
 
 
 
 2 
 
 
 
 
 0 
 
 
 
 
 0.5 
 
 
 
 
 1 
 
 
 
 
 1.5 
 
 
 
 
 2 
 
 
 
 
 2.5 
 
 
 
 
 3 
 
 
 
 
 3.5 
 
 
 
 
 4 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 L/W 
 
 
 
 
 t 
 
 
 
 
 eta0 = 0.1 
 
 
 
 
 eta 0 = 0.1 
 
 
 
	<path stroke='gray' d='M371.4,62.1 L424.8,62.1 M77.8,296.5 L78.4,296.5 L79.0,296.5 L79.5,296.5 L80.1,296.5 L80.7,296.5
 L81.3,296.6 L81.9,296.6 L82.5,296.6 L83.0,296.6 L83.6,296.7 L84.2,296.7 L84.8,296.7 L85.4,296.8
 L86.0,296.8 L86.5,296.9 L87.1,296.9 L87.7,297.0 L88.3,297.1 L88.9,297.1 L89.5,297.2 L90.0,297.3
 L90.6,297.3 L91.2,297.4 L91.8,297.5 L92.4,297.6 L93.0,297.7 L93.5,297.8 L94.1,297.9 L94.7,298.0
 L95.3,298.1 L95.9,298.2 L96.5,298.3 L97.0,298.4 L97.6,298.5 L98.2,298.7 L98.8,298.8 L99.4,298.9
 L99.9,299.1 L100.5,299.2 L101.1,299.3 L101.7,299.5 L102.3,299.6 L102.9,299.8 L103.4,299.9 L104.0,300.1
 L104.6,300.3 L105.2,300.4 L105.8,300.6 L106.4,300.8 L106.9,301.0 L107.5,301.1 L108.1,301.3 L108.7,301.5
 L109.3,301.7 L109.9,301.9 L110.4,302.1 L111.0,302.3 L111.6,302.5 L112.2,302.7 L112.8,303.0 L113.4,303.2
 L113.9,303.4 L114.5,303.6 L115.1,303.9 L115.7,304.1 L116.3,304.3 L116.9,304.6 L117.4,304.8 L118.0,305.1
 L118.6,305.3 L119.2,305.6 L119.8,305.8 L120.3,306.1 L120.9,306.3 L121.5,306.6 L122.1,306.9 L122.7,307.1
 L123.3,307.4 L123.8,307.7 L124.4,308.0 L125.0,308.3 L125.6,308.5 L126.2,308.8 L126.8,309.1 L127.3,309.4
 L127.9,309.7 L128.5,310.0 L129.1,310.3 L129.7,310.6 L130.3,310.9 L130.8,311.2 L131.4,311.6 L132.0,311.9
 L132.6,312.2 L133.2,312.5 L133.8,312.8 L134.3,313.1 L134.9,313.5 L135.5,313.8 L136.1,314.1 L136.7,314.4
 L137.3,314.7 L137.8,315.1 L138.4,315.4 L139.0,315.7 L139.6,316.1 L140.2,316.4 L140.8,316.7 L141.3,317.1
 L141.9,317.4 L142.5,317.7 L143.1,318.1 L143.7,318.4 L144.2,318.7 L144.8,319.1 L145.4,319.4 L146.0,319.7
 L146.6,320.0 L147.2,320.4 L147.7,320.7 L148.3,321.0 L148.9,321.4 L149.5,321.7 L150.1,322.0 L150.7,322.4
 L151.2,322.7 L151.8,323.0 L152.4,323.3 L153.0,323.7 L153.6,324.0 L154.2,324.3 L154.7,324.6 L155.3,324.9
 L155.9,325.3 L156.5,325.6 L157.1,325.9 L157.7,326.2 L158.2,326.5 L158.8,326.8 L159.4,327.1 L160.0,327.4
 L160.6,327.7 L161.2,328.0 L161.7,328.3 L162.3,328.6 L162.9,328.9 L163.5,329.2 L164.1,329.5 L164.6,329.8
 L165.2,330.0 L165.8,330.3 L166.4,330.6 L167.0,330.9 L167.6,331.1 L168.1,331.4 L168.7,331.7 L169.3,331.9
 L169.9,332.2 L170.5,332.5 L171.1,332.7 L171.6,333.0 L172.2,333.2 L172.8,333.4 L173.4,333.7 L174.0,333.9
 L174.6,334.2 L175.1,334.4 L175.7,334.6 L176.3,334.8 L176.9,335.1 L177.5,335.3 L178.1,335.5 L178.6,335.7
 L179.2,335.9 L179.8,336.1 L180.4,336.3 L181.0,336.5 L181.6,336.7 L182.1,336.9 L182.7,337.1 L183.3,337.3
 L183.9,337.4 L184.5,337.6 L185.0,337.8 L185.6,337.9 L186.2,338.1 L186.8,338.3 L187.4,338.4 L188.0,338.6
 L188.5,338.7 L189.1,338.9 L189.7,339.0 L190.3,339.1 L190.9,339.3 L191.5,339.4 L192.0,339.5 L192.6,339.7
 L193.2,339.8 L193.8,339.9 L194.4,340.0 L195.0,340.1 L195.5,340.2 L196.1,340.3 L196.7,340.4 L197.3,340.5
 L197.9,340.6 L198.5,340.7 L199.0,340.8 L199.6,340.8 L200.2,340.9 L200.8,341.0 L201.4,341.1 L202.0,341.1
 L202.5,341.2 L203.1,341.2 L203.7,341.3 L204.3,341.3 L204.9,341.4 L205.4,341.4 L206.0,341.5 L206.6,341.5
 L207.2,341.5 L207.8,341.6 L208.4,341.6 L208.9,341.6 L209.5,341.6 L210.1,341.6 L210.7,341.6 L211.3,341.7
 L211.9,341.7 L212.4,341.7 L213.0,341.7 L213.6,341.6 L214.2,341.6 L214.8,341.6 L215.4,341.6 L215.9,341.6
 L216.5,341.6 L217.1,341.5 L217.7,341.5 L218.3,341.5 L218.9,341.4 L219.4,341.4 L220.0,341.4 L220.6,341.3
 L221.2,34